In [ ]:
import torch
from torch import nn
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
from rt_whisper.models import BoundaryWordFilter

In [ ]:
EXPONENT = 2
MODELS = {

    "1차. 16k 경계 head": "/workspaces/dev/test/optimize/all/.cache/20250901/1차_기록/step2_16b-96k-model-3090/001_0_068_head_model.pth",
    "1차. 16k 경계 tail": "/workspaces/dev/test/optimize/all/.cache/20250901/1차_기록/step2_16b-96k-model-3090/001_0_068_tail_model.pth",

    "2차. 32k 경계 head": "/workspaces/dev/test/optimize/all/.cache/20250901/2차_기록/step2_16b-96k-model-3090/001_0_067_head_model.pth",
    "2차. 16k 경계 tail": "/workspaces/dev/test/optimize/all/.cache/20250901/2차_기록/step2_16b-96k-model-3090/001_0_067_tail_model.pth",

    "3차. WER 경계 head": "/workspaces/dev/test/optimize/all/.cache/20250901/3차_기록/step3_16b-96k-3090/001_389_0_head_model.pth",
    "3차. WER 경계 tail": "/workspaces/dev/test/optimize/all/.cache/20250901/3차_기록/step3_16b-96k-3090/001_389_0_tail_model.pth",

    "cp-head": "/workspaces/dev/test/optimize/all/hyperparameters/20250903/3s/cp/step2_3s-96k-cp-model/001_401_0_head_model.pth",
    "cp-tail": "/workspaces/dev/test/optimize/all/hyperparameters/20250903/3s/cp/step2_3s-96k-cp-model/001_401_0_tail_model.pth",

    "cpm-head": "/workspaces/dev/test/optimize/all/hyperparameters/20250903/3s/cpm/step2_3s-96k-cpm-model/001_405_0_head_model.pth",
    "cpm-tail": "/workspaces/dev/test/optimize/all/hyperparameters/20250903/3s/cpm/step2_3s-96k-cpm-model/001_405_0_tail_model.pth",

    "oc-head": "/workspaces/dev/test/optimize/all/hyperparameters/20250903/3s/oc/step2_3s-96k-oc-model/001_403_0_head_model.pth",
    "oc-tail": "/workspaces/dev/test/optimize/all/hyperparameters/20250903/3s/oc/step2_3s-96k-oc-model/001_403_0_tail_model.pth",

    "op-head": "/workspaces/dev/test/optimize/all/hyperparameters/20250903/3s/op/step2_3s-96k-op-model/001_405_0_head_model.pth",
    "op-tail": "/workspaces/dev/test/optimize/all/hyperparameters/20250903/3s/op/step2_3s-96k-op-model/001_405_0_tail_model.pth",

}

In [ ]:
models = {k: Path(v) for k, v in MODELS.items()}

if not all(m.exists() for m in models.values()):
    raise FileNotFoundError(f"One or more model files do not exist: {models}")

In [ ]:
b_models = {k: BoundaryWordFilter.load(v) for k, v in models.items()}

In [ ]:
def generate_data(size:int, boundary:int = 16000, dur:int = None):
    for _ in range(size):
        start, end = sorted(np.random.randint(0, boundary, 2))
        if dur is not None:
            if start + dur <= boundary:
                end = start + dur
            else:
                start = end - dur

        start = start // 160 * 160
        end = end // 160 * 160
        mid = (start + end) / 2
        dur = end - start
        weight = (mid/boundary) ** EXPONENT

        start /= boundary
        end /= boundary
        mid /= boundary
        dur /= boundary

        yield (start, end, mid, dur), weight

def plot_mid_vs_pred_and_target(
    model: nn.Module,
    X: torch.Tensor,
    Y: torch.Tensor,
    device: str | torch.device = "cpu"
):
    model.to(device)
    X = X.to(device)
    Y = Y.to(device)

    with torch.no_grad():
        # mid만 추출 (X의 세 번째 컬럼)
        mids = X[:, 2].cpu().numpy()

        # 모델 예측
        preds = model(X).cpu().numpy().flatten()
        Y = Y.cpu().numpy().flatten()

    # 산점도 그리기
    plt.figure(figsize=(8,5))
    plt.scatter(mids, Y, alpha=0.5, label="Target (Y)", color="blue")
    plt.scatter(mids, preds, alpha=0.5, label="Model Output", color="red")
    plt.xlabel("mid")
    plt.ylabel("value")
    plt.title("Mid vs Model Prediction & Target")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()

In [ ]:
datasets = {dur:[data for data in generate_data(1024, dur=dur)] for dur in range(0, 16000, 1600)}

In [ ]:
def plot_with_dur(model:nn.Module, device: str|torch.device = "cpu"):
    for dur, data in datasets.items():
        print(f"========================= {dur} ===============================")
        X, Y = zip(*data)
        X = torch.tensor(X, dtype=torch.float32)
        Y = torch.tensor(Y, dtype=torch.float32).reshape(-1, 1)

        plot_mid_vs_pred_and_target(model, X, Y, device)
        break

In [ ]:
for name, model in b_models.items():
    print(name)
    plot_with_dur(model, "cuda")